# Phase 2 — Basic VGGT Inference Pipeline

This notebook implements the full VGGT pipeline for 3-D reconstruction:

1. Environment setup (Colab / Kaggle)
2. GPU check & model loading
3. Load images
4. Run VGGT inference
5. Build & visualise 3-D point cloud
6. Visualise depth maps & confidence
7. Save COLMAP sparse reconstruction
8. Performance benchmarks (memory, time vs. frame count)
9. Qualitative comparison summary

**Target hardware:** Colab / Kaggle free-tier T4 (16 GB VRAM)

## 1. Environment Setup

In [ ]:
import sys, os, subprocess

# Detect platform
IN_COLAB  = 'google.colab' in sys.modules
IN_KAGGLE = os.path.exists('/kaggle')
print(f'Colab: {IN_COLAB}  |  Kaggle: {IN_KAGGLE}')

if IN_COLAB:
    # Colab: install everything from scratch
    for p in [
        'numpy==1.26.4',
        'torch torchvision --index-url https://download.pytorch.org/whl/cu118',
        'plotly matplotlib open3d trimesh huggingface_hub',
        'git+https://github.com/jytime/LightGlue.git',
    ]:
        subprocess.run(f'pip install -q {p}', shell=True)

elif IN_KAGGLE:
    # Kaggle ships torch + numpy pre-built and ABI-compatible.
    # Installing numpy==1.x would break the pre-compiled torch binaries.
    # Only add truly missing packages.
    for p in [
        'plotly open3d trimesh huggingface_hub',
        'git+https://github.com/jytime/LightGlue.git',
    ]:
        subprocess.run(f'pip install -q {p}', shell=True)

if IN_COLAB or IN_KAGGLE:
    # Clone VGGT if not already present
    if not os.path.exists('vggt'):
        subprocess.run('git clone --depth 1 https://github.com/facebookresearch/vggt.git', shell=True)

    # Install vggt deps; on Kaggle, pin numpy/torch to pre-installed versions
    # so vggt/requirements.txt can't downgrade them and break the ABI.
    if IN_KAGGLE:
        np_ver = subprocess.check_output(
            ['python', '-c', 'import numpy; print(numpy.__version__)'], text=True
        ).strip()
        with open('/tmp/kaggle_pin.txt', 'w') as _f:
            _f.write(f'numpy=={np_ver}\n')
        subprocess.run('pip install -q -r vggt/requirements.txt -c /tmp/kaggle_pin.txt', shell=True)
    else:
        subprocess.run('pip install -q -r vggt/requirements.txt', shell=True)

    # Make the vggt package importable in this kernel without needing a restart
    vggt_root = os.path.abspath('vggt')
    if vggt_root not in sys.path:
        sys.path.insert(0, vggt_root)

    # Clone this eval repo if running directly in Colab/Kaggle
    if not os.path.exists('vggt-eval'):
        subprocess.run('git clone --depth 1 https://github.com/insha-py/vggt-eval.git', shell=True)
    eval_root = os.path.abspath('vggt-eval')
    if eval_root not in sys.path:
        sys.path.insert(0, eval_root)

# Quick import smoke-test
try:
    from vggt.models.vggt import VGGT
    print('vggt package found OK')
except ModuleNotFoundError as e:
    print(f'WARNING: {e} — check that the vggt clone succeeded above.')

print('Setup complete.')

In [ ]:
import torch

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'GPU : {props.name}')
    print(f'VRAM: {props.total_memory / 1024**3:.1f} GB')
    print(f'CUDA capability: {props.major}.{props.minor}')
else:
    print('WARNING: No GPU found — inference will be very slow on CPU.')

## 2. Imports & Configuration

In [ ]:
import sys, os
import numpy as np
import matplotlib.pyplot as plt

# Add repo root to path
REPO_ROOT = os.path.abspath(os.path.join(os.path.dirname(''), '..'))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from src.pipeline      import VGGTPipeline, print_result_summary
from src.visualization import (
    plot_point_cloud, plot_depth_maps, plot_cameras,
    plot_trajectory, plot_memory_vs_frames, save_ply, load_ply,
)
from src.metrics import (
    compute_ate, compute_rpe, compute_rotation_errors,
    compute_auc, MemoryProfiler, Timer, benchmark_inference,
    print_metrics_table,
)

# ── Configuration ──────────────────────────────────────────────
# Update this path to point to a folder of images.
# If no real data is available, leave as None to use synthetic images.
IMAGE_DIR    = None         # e.g. '/kaggle/input/my-scene/images'
OUTPUT_DIR   = '../results'
CHECKPOINT   = None         # local path, or None to auto-download
CONF_THRESH  = 5.0          # depth-confidence threshold
MAX_FRAMES   = 20           # limit frames to stay within 16 GB VRAM

os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Imports OK.')

## 3. Prepare Sample Images

If `IMAGE_DIR` is None we generate a small set of synthetic images so the
notebook can run end-to-end without real data.

In [ ]:
import glob
from PIL import Image as PILImage

def make_synthetic_images(out_dir, n=12, size=256):
    """Create synthetic gradient images for smoke-testing."""
    os.makedirs(out_dir, exist_ok=True)
    for i in range(n):
        arr = np.zeros((size, size, 3), dtype=np.uint8)
        # Horizontal gradient with slight rotation offset per frame
        x = np.linspace(0, 255, size, dtype=np.uint8)
        arr[:, :, 0] = x[None, :]
        arr[:, :, 1] = np.roll(x, i * 5)[None, :]
        arr[:, :, 2] = np.roll(x, i * 10)[:, None]
        PILImage.fromarray(arr).save(os.path.join(out_dir, f'frame_{i:04d}.png'))
    return out_dir

if IMAGE_DIR is None:
    print('No IMAGE_DIR provided — generating synthetic test images.')
    IMAGE_DIR = make_synthetic_images('../datasets/synthetic/test_scene', n=MAX_FRAMES)

paths = sorted(glob.glob(os.path.join(IMAGE_DIR, '*')))
paths = [p for p in paths if p.lower().endswith(('.jpg', '.jpeg', '.png'))]
print(f'Found {len(paths)} images in {IMAGE_DIR}')

# Preview first 4 frames
fig, axes = plt.subplots(1, min(4, len(paths)), figsize=(12, 3))
for ax, p in zip(np.array(axes).flat, paths[:4]):
    ax.imshow(PILImage.open(p))
    ax.set_title(os.path.basename(p), fontsize=8)
    ax.axis('off')
plt.suptitle('Input frames (first 4)')
plt.tight_layout()
plt.show()

## 4. Load VGGT Model

In [ ]:
pipe = VGGTPipeline()
pipe.load_model(checkpoint_path=CHECKPOINT)
print('Model loaded successfully.')

## 5. Run Inference

In [ ]:
result = pipe.run(
    image_dir=IMAGE_DIR,
    max_frames=MAX_FRAMES,
    conf_thresh=CONF_THRESH,
)

print_result_summary(result)

## 6. Point Cloud Visualisation

In [ ]:
fig = plot_point_cloud(
    result['point_cloud'],
    colors=result['point_colors'],
    title=f'VGGT Reconstruction ({len(result["point_cloud"]):,} points)',
    save_html=os.path.join(OUTPUT_DIR, 'figures', 'point_cloud.html'),
    show=True,
)

In [ ]:
fig_cam = plot_cameras(
    result['extrinsic'],
    points=result['point_cloud'],
    point_colors=result['point_colors'],
    frustum_scale=0.05,
    title='Camera Poses + Point Cloud',
    save_html=os.path.join(OUTPUT_DIR, 'figures', 'cameras.html'),
    show=True,
)

## 7. Depth Map Visualisation

In [ ]:
# Show the first min(8, N) frames
n_show = min(8, len(result['depth_map']))
fig_d = plot_depth_maps(
    result['depth_map'][:n_show],
    conf_maps=result['depth_conf'][:n_show],
    n_cols=4,
    title='Depth & Confidence Maps',
    save_path=os.path.join(OUTPUT_DIR, 'figures', 'depth_maps.png'),
    show=True,
)

## 8. Save Outputs

In [ ]:
# Save point cloud as PLY
ply_path = os.path.join(OUTPUT_DIR, 'point_cloud.ply')
pipe.save_ply(result, ply_path)

# Save COLMAP sparse reconstruction (requires pycolmap + trimesh)
try:
    pipe.save_colmap(
        result, IMAGE_DIR,
        output_dir=os.path.join(OUTPUT_DIR, 'sparse'),
    )
except Exception as e:
    print(f'COLMAP export skipped: {e}')

print('Outputs saved.')

## 9. Performance Benchmarking

Measure GPU memory and inference time as a function of frame count.
This reproduces / extends Table 9 from the VGGT paper for your hardware.

In [ ]:
from src.pipeline import load_images_from_list, run_vggt_inference
from src.metrics  import MemoryProfiler, Timer

# Frame counts to test (stop before OOM on your GPU)
# T4 (16 GB): safe up to ~40-50 frames; P100 (16 GB): similar
FRAME_COUNTS = [1, 5, 10, 20, 30]  # extend carefully

bench_results = []
for n in FRAME_COUNTS:
    if n > len(paths):
        print(f'Skipping n={n}: not enough images')
        continue
    imgs, _ = load_images_from_list(paths[:n])

    with MemoryProfiler() as mem, Timer() as t:
        run_vggt_inference(pipe.model, imgs, pipe.device, pipe.dtype)

    row = {
        'n_frames':    n,
        'time_mean_s': t.elapsed,
        'time_std_s':  0.0,
        'peak_gpu_mb': mem.peak_mb,
    }
    bench_results.append(row)
    print(f'  n={n:3d}  time={t.elapsed:.2f}s  peak={mem.peak_mb:.0f} MB')

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
if bench_results:
    fig_bm = plot_memory_vs_frames(
        bench_results,
        title=f'VGGT Benchmark on {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}',
        save_path=os.path.join(OUTPUT_DIR, 'figures', 'benchmark_memory.png'),
        show=True,
    )

    # Print table
    print(f'\n{"─"*55}')
    print(f'  {"N frames":>10}  {"Time (s)":>10}  {"Peak (MB)":>12}  {"Peak (GB)":>10}')
    print(f'{"─"*55}')
    for r in bench_results:
        print(f'  {r["n_frames"]:>10d}  {r["time_mean_s"]:>10.2f}  '
              f'{r["peak_gpu_mb"]:>12.0f}  {r["peak_gpu_mb"]/1024:>10.2f}')
    print(f'{"─"*55}')

## 10. Camera Pose Quality (Self-Consistency Check)

Without ground-truth, we measure **relative pose consistency**: the residual
when reprojecting consecutive camera centres from predicted depth.

In [ ]:
from src.metrics import compute_rpe, compute_rotation_errors

# Self-consistency: compare result with itself (trivially zero)
# For real evaluation, supply ground-truth extrinsics here.
# Example: gt_extrs = np.load('path/to/gt_poses.npy')  # (N,3,4)
gt_extrs = None

if gt_extrs is not None:
    ate = compute_ate(result['extrinsic'], gt_extrs, align=True)
    rpe = compute_rpe(result['extrinsic'], gt_extrs)
    auc = compute_auc(result['extrinsic'], gt_extrs)
    print_metrics_table(ate, 'ATE (metres)')
    print_metrics_table(rpe, 'RPE')
    print_metrics_table(auc, 'AUC @ rotation thresholds')
else:
    print('No ground-truth poses available — skipping quantitative evaluation.')
    print('Supply gt_extrs as (N,3,4) numpy array to enable.')

# Trajectory plot
fig_traj = plot_trajectory(
    result['extrinsic'],
    gt_extrinsics=gt_extrs,
    title='Camera Trajectory (top-down XZ view)',
    save_path=os.path.join(OUTPUT_DIR, 'figures', 'trajectory.png'),
    show=True,
)

## 11. Summary

| Metric | Value |
|--------|-------|
| Frames processed | — |
| Points in cloud  | — |
| Inference time   | — |
| Peak GPU memory  | — |

Fill in the table above from the numbers printed in the cells above.

### Next Steps
- Run on longer sequences → see memory constraints → go to **Notebook 02**
- Compare qualitatively with COLMAP output
- Add ground-truth poses for quantitative ATE/RPE/AUC evaluation

In [ ]:
if bench_results:
    print('=== Phase 2 Summary ===')
    print(f'  Image directory : {IMAGE_DIR}')
    print(f'  Frames          : {len(result["extrinsic"])}')
    print(f'  Points          : {len(result["point_cloud"]):,}')
    print(f'  Inference time  : {result["inference_time_s"]:.2f} s')
    print(f'  Peak GPU memory : {result["peak_gpu_mb"]:.0f} MB ({result["peak_gpu_mb"]/1024:.2f} GB)')
    print(f'  PLY saved to    : {ply_path}')